**Programmer:** python_scripts (Abhijith Warrier)

**PYTHON SCRIPT TO _EXPLORE HOW DECORATORS REALLY WORK UNDER THE HOOD — USING CLOSURES, FUNCTION WRAPPING & functools.wraps_. 🐍🧠**

Decorators look simple, but under the hood they combine:
- **Closures** (functions carrying enclosed state)
- **Function wrapping** (returning a new function)
- **@decorator syntax** (syntactic sugar)
- **functools.wraps** (preserving metadata like name & docstring)

This DeepCut reveals exactly how decorators work.

---

## 📦 Import Standard Library

In [1]:
import functools    # for preserving metadata of wrapped functions
import inspect      # to inspect signatures and metadata

---

## 🧩 Snippet 1 — A decorator is just a function that takes a function and returns a function

This is the simplest version:
- Input → a function
- Output → another function
- Usually implemented using a **closure**

In [2]:
def simple_decorator(func):
    def wrapper():
        print("Before call")
        func()
        print("After call")
    return wrapper

def greet():
    print("Hello!")

decorated = simple_decorator(greet)
decorated()

Before call
Hello!
After call


---

## 🔍 Snippet 2 — The @decorator syntax expands to function = decorator(function)

Python rewrites:

```python
@decorator
def f():
    ...
```

as:

```python
def f():
    ...
f = decorator(f)
```

In [3]:
def announce(func):
    def wrapper():
        print("Announcing...")
        return func()
    return wrapper

@announce              # equivalent to greet = announce(greet)
def greet():
    print("Hi from greet()")

greet()

Announcing...
Hi from greet()


---

## 🧠 Snippet 3 — Decorators can accept arguments (requires 3 layers)

- Outer layer → decorator arguments
- Middle layer → receives function
- Inner layer → wrapper executed at call time

In [4]:
def repeat(n):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(n):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def hello():
    print("Hello!")

hello()

Hello!
Hello!
Hello!


---

## ⚠️ Snippet 4 — Without wraps, decorators break metadata

Wrapped functions lose:
- `__name__`
- `__doc__`
- Signature information

This causes issues with debugging, introspection, frameworks, and IDEs.

In [5]:
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def square(x):
    """Return x squared."""
    return x * x

print(square.__name__)   # wrong: "wrapper"
print(square.__doc__)    # lost!
print(inspect.signature(square))   # loses signature

wrapper
None
(*args, **kwargs)


---

## ✨ Snippet 5 — wraps fixes metadata, making decorated functions act like originals

`wraps` copies over:
- function name
- docstring
- annotations
- module
- signature

In [6]:
def good_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def square(x):
    """Return x squared."""
    return x * x

print(square.__name__)           # correct: "square"
print(square.__doc__)            # preserved
print(inspect.signature(square)) # real signature

square
Return x squared.
(x)


---

## ⏱️ Snippet 6 — A practical decorator that times a function

Useful example showing how decorators wrap behavior before/after the target function.

In [7]:
import time

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        duration = time.perf_counter() - start
        print(f"{func.__name__} took {duration:.6f}s")
        return result
    return wrapper

@timer
def compute():
    sum(i*i for i in range(10_000))

compute()

compute took 0.001011s


---

## 🧱 Snippet 7 — Multiple decorators stack, bottom-up

The order changes behavior:
```python
@A
@B
def f(): ...
```

Becomes:

```python
f = A(B(f))
```

In [8]:
def A(func):
    @functools.wraps(func)
    def wrapper():
        print("A start")
        func()
        print("A end")
    return wrapper

def B(func):
    @functools.wraps(func)
    def wrapper():
        print("B start")
        func()
        print("B end")
    return wrapper

@A
@B
def hello():
    print("Hello!")

hello()

A start
B start
Hello!
B end
A end


---

## ✅ One-liner Takeaway

**Decorators work by wrapping functions with closures — and `functools.wraps` is essential to keep your wrapped functions looking like the originals.**

---